# Phase 2 grounding: how much does the scoring protocol move a fixed denoiser's PSNR? (CPU, released outputs only)

The audit found that GRL-B's published numbers are computed on cropped images, while DMID's and ours use full ones:
* BSD68 / CBSD68 drop the last row and column;
* McMaster uses the top-left 496×496;
* Kodak24 uses the central 500×500 crop cut to 496×496.

This notebook holds the denoiser fixed and changes only the scoring protocol:
1. DMID-d's released full-image outputs (CBSD68 / Kodak24 / McMaster / Urban100, σ 15 / 25 / 50) are scored under
   seven protocols: full image; GRL's crop for each set; top-left multiple of 8; top-left multiple of 16; 8-px border
   shave; Y channel (BT.601, MATLAB `rgb2ycbcr` range); and GRL's crop on the Y channel.
2. DMID-d and GRL-B are compared on exactly the same pixels, GRL's crop region (different noise draws), and that
   comparison is set against the comparison of published numbers.
Kaggle: CPU, internet on.

In [ ]:
import io, os, re, tarfile, time, urllib.request, zipfile
import numpy as np
import pandas as pd
from PIL import Image

OUT, CACHE = "outputs", "/tmp/cache"
for d in (OUT, CACHE):
    os.makedirs(d, exist_ok=True)
T0 = time.time()


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        with urllib.request.urlopen(url, timeout=600) as r, open(path, "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    return path


GH = "https://github.com/{}/releases/download/{}/{}"
dmid_zip = download(GH.format("Li-Tong-621/DMID", "v1.0", "1-Classical_gaussion.zip"), f"{CACHE}/dmid_results.zip")
grl_tgz = download(GH.format("ofsoundof/GRL-Image-Restoration", "v1.0.0", "visual_results_dn.tar.gz"), f"{CACHE}/grl_dn.tar.gz")
print(f"downloads done ({(time.time() - T0) / 60:.1f} min)", flush=True)

## DMID-d released outputs and their clean images (both in DMID's archive)

In [ ]:
SHORT = {"CBSD68": "CBSD68", "Kodak": "Kodak24", "McMaster": "McMaster", "Urban100": "Urban100"}
DMID = {}   # (dataset, sigma, idx) -> (denoised uint8, clean uint8)
with zipfile.ZipFile(dmid_zip) as z:
    names = z.namelist()
    for n in names:
        m = re.match(r"1-Classical_gaussion/(\w+)/(\d+)/denoise/(\d+)_denoise\.png$", n)
        if m and m.group(1) in SHORT:
            short, sigma, idx = m.group(1), int(m.group(2)), int(m.group(3))
            clean = np.asarray(Image.open(io.BytesIO(z.read(f"1-Classical_gaussion/{short}/15/clean/{idx}_clean.png"))).convert("RGB"))
            den = np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))
            DMID[(SHORT[short], sigma, idx)] = (den, clean)
print(len(DMID), "DMID outputs", flush=True)


def ycbcr_y(rgb):
    """MATLAB rgb2ycbcr luma on 0-255 values: 16 + (65.481 R + 128.553 G + 24.966 B) / 255 (rgb in 0-255)."""
    r = rgb.astype(np.float64)
    return 16.0 + (65.481 * r[..., 0] + 128.553 * r[..., 1] + 24.966 * r[..., 2]) / 255.0


def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 10 * np.log10(255 ** 2 / mse)


def grl_crop(ds, h, w):
    """The pixel region GRL-B's released outputs cover, as found by `grl-outputs-audit` / `grl-outputs-audit-kodak`:
    the top-left crop to a multiple of 8, after a central 500×500 crop for Kodak24."""
    if ds == "Kodak24":
        i, j = (h - 500) // 2, (w - 500) // 2
        return slice(i, i + 496), slice(j, j + 496)
    return slice(0, h // 8 * 8), slice(0, w // 8 * 8)


PROTOCOLS = {
    "full": lambda ds, h, w: (slice(0, h), slice(0, w)),
    "GRL crop": grl_crop,
    "top-left ×8": lambda ds, h, w: (slice(0, h // 8 * 8), slice(0, w // 8 * 8)),
    "top-left ×16": lambda ds, h, w: (slice(0, h // 16 * 16), slice(0, w // 16 * 16)),
    "shave 8": lambda ds, h, w: (slice(8, h - 8), slice(8, w - 8)),
}
rows = []
for (ds, sigma, idx), (den, clean) in DMID.items():
    h, w = clean.shape[:2]
    for p, f in PROTOCOLS.items():
        si, sj = f(ds, h, w)
        rows.append(dict(method="DMID-d", dataset=ds, sigma=sigma, idx=idx, protocol=p, channel="RGB", psnr=psnr(den[si, sj], clean[si, sj])))
    for p in ("full", "GRL crop"):
        si, sj = PROTOCOLS[p](ds, h, w)
        rows.append(dict(method="DMID-d", dataset=ds, sigma=sigma, idx=idx, protocol=p, channel="Y",
                         psnr=psnr(ycbcr_y(den[si, sj]), ycbcr_y(clean[si, sj]))))
P = pd.DataFrame(rows)
P.to_csv(f"{OUT}/dmid_protocols.csv", index=False)
M = P.groupby(["dataset", "sigma", "channel", "protocol"]).psnr.mean().unstack(["channel", "protocol"])
print("DMID-d mean PSNR by protocol")
print(M.round(3).to_string())
D = M.sub(M[("RGB", "full")], axis=0)
print("\nshift against the full-image RGB score (dB)")
print(D.round(3).to_string(), flush=True)

## DMID-d against GRL-B on the same pixels (GRL's crop region; each method on its own noise draws)

In [ ]:
CLEAN = {}
for (ds, sigma, idx), (den, clean) in DMID.items():
    CLEAN.setdefault(ds, {})[idx] = clean
GRL = {}
with tarfile.open(grl_tgz) as t:
    for m in t.getmembers():
        mm = re.search(r"base_color/Sigma(\d+)/(CBSD68|Kodak24|McMaster|Urban100)/([^/]+)$", m.name)
        if m.isfile() and mm:
            GRL[(mm.group(2), int(mm.group(1)), mm.group(3))] = np.asarray(Image.open(io.BytesIO(t.extractfile(m).read())).convert("RGB"))
print(len(GRL), "GRL-B colour outputs", flush=True)


def match(out, ds):
    """Clean image and orientation for one GRL output, by MSE on GRL's crop region."""
    best = (np.inf, None, None)
    for idx, clean in CLEAN[ds].items():
        for k in range(4):
            c = np.rot90(clean, k)
            h, w = c.shape[:2]
            si, sj = grl_crop(ds, h, w)
            reg = c[si, sj]
            if reg.shape != out.shape:
                continue
            mse = np.mean((out.astype(np.float64) - reg.astype(np.float64)) ** 2)
            if mse < best[0]:
                best = (mse, idx, k)
    return best


MAP = {}
for (ds, sigma, name), out in GRL.items():
    if sigma == 15:
        mse, idx, k = match(out, ds)
        MAP[(ds, name)] = (idx, k, 10 * np.log10(255 ** 2 / mse))
bad = {k: v for k, v in MAP.items() if v[2] < 25}
print(len(MAP), "GRL images matched;", len(bad), "suspicious (< 25 dB at σ 15):", list(bad.items())[:5], flush=True)
rows = []
for (ds, sigma, name), out in GRL.items():
    idx, k, _ = MAP[(ds, name)]
    den, clean = DMID[(ds, sigma, idx)]
    c, dd = np.rot90(clean, k), np.rot90(den, k)
    si, sj = grl_crop(ds, *c.shape[:2])
    rows.append(dict(dataset=ds, sigma=sigma, idx=idx, grl=psnr(out, c[si, sj]), dmid_same_pixels=psnr(dd[si, sj], c[si, sj]),
                     dmid_full=psnr(den, clean)))
C = pd.DataFrame(rows)
C.to_csv(f"{OUT}/dmid_vs_grl.csv", index=False)
PUB = {("CBSD68", 15): (34.45, 34.45), ("CBSD68", 25): (31.82, 31.86), ("CBSD68", 50): (28.62, 28.72),
       ("Kodak24", 15): (35.43, 35.51), ("Kodak24", 25): (33.02, 33.12), ("Kodak24", 50): (29.93, 30.14),
       ("McMaster", 15): (35.73, 35.72), ("McMaster", 25): (33.46, 33.49), ("McMaster", 50): (30.36, 30.50),
       ("Urban100", 15): (35.54, 35.26), ("Urban100", 25): (33.35, 33.11), ("Urban100", 50): (30.46, 30.28)}
S = C.groupby(["dataset", "sigma"]).agg(n=("grl", "size"), grl=("grl", "mean"), dmid_same_pixels=("dmid_same_pixels", "mean"),
                                        dmid_full=("dmid_full", "mean")).reset_index()
S["gap_same_pixels"] = S.dmid_same_pixels - S.grl
S["gap_as_published"] = [PUB[(d, s)][1] - PUB[(d, s)][0] for d, s in zip(S.dataset, S.sigma)]
S["published_grl"] = [PUB[(d, s)][0] for d, s in zip(S.dataset, S.sigma)]
S["published_dmid"] = [PUB[(d, s)][1] for d, s in zip(S.dataset, S.sigma)]
S["rank_flip"] = np.sign(S.gap_same_pixels.round(3)) != np.sign(S.gap_as_published.round(3))
S.to_csv(f"{OUT}/dmid_vs_grl_summary.csv", index=False)
print(S.round(3).to_string(index=False))
print(f"done in {(time.time() - T0) / 60:.1f} min")